# PSDCodec Deployment Animation Demo

This notebook is a thin deployment walkthrough built around the exported `models/exports/demo` artifacts. It evaluates a batch of raw campaign frames, selects representative examples across the distortion range, and renders a `matplotlib.animation.FuncAnimation` that cycles through many deployment examples.


## Notebook Flow

1. Load the exported deployment artifacts and restore the ONNX encoder + PyTorch decoder boundary.
2. Evaluate a configurable batch of campaign frames through the deployed codec.
3. Summarize deployment readiness and select a representative animation subset.
4. Render a `FuncAnimation` showing original, preprocessing-only, and codec PSDs for many examples.


In [16]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
src_root = project_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from interfaces.demo_animation import (
    build_animation_frame_summary_rows,
    create_deployment_animation,
    select_animation_frame_reports,
)
from interfaces.deployment import create_deployment_service, evaluate_deployment_batch

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#FCFCFD",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})


In [17]:
export_dir = project_root / "models" / "exports" / "demo"
summary_path = export_dir / "training_summary.json"
if not summary_path.exists():
    raise FileNotFoundError(
        "Expected a completed demo export at models/exports/demo. Finish the canonical "
        "training run before opening this notebook."
    )

training_summary = json.loads(summary_path.read_text(encoding="utf-8"))
service, artifacts = create_deployment_service(export_dir)

best_epoch_index = training_summary["best_epoch_index"]
best_score = training_summary["best_selection_score"]
display(Markdown(
    f"**Loaded export:** `{export_dir}`\n\n"
    f"**Best checkpoint epoch:** {best_epoch_index + 1}\n\n"
    f"**Best deployment score:** {best_score:.6f}"
))
artifacts.checkpoint_path


FileNotFoundError: Expected a completed demo export at models/exports/demo. Finish the canonical training run before opening this notebook.

In [ ]:
batch_frame_count = 48
animation_frame_count = 24

batch_report = evaluate_deployment_batch(
    service,
    artifacts,
    max_frames=batch_frame_count,
    task_config=artifacts.experiment_config.task,
)

summary_frame = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in batch_report.summary.to_display_dict().items()]
)
display(summary_frame)

display(Markdown(
    f"**Readiness verdict:** `{batch_report.assessment.verdict}`\n\n"
    + "\n".join(f"- {reason}" for reason in batch_report.assessment.reasons)
))


In [ ]:
animation_reports = select_animation_frame_reports(
    batch_report,
    frame_count=animation_frame_count,
)

selection_frame = pd.DataFrame(build_animation_frame_summary_rows(animation_reports))
selection_frame


## Animated Deployment Walkthrough

The animation below cycles through representative deployment examples across the evaluated distortion range. The top panel shows the current PSD reconstruction, the bottom-left panel places the current frame in the packet-size / distortion distribution, and the bottom-right panel prints the frame-local deployment metrics.


In [ ]:
interval_ms = 900
animation = create_deployment_animation(
    animation_reports,
    interval_ms=interval_ms,
    show_noise_floor=True,
)
animation_html = HTML(animation.to_jshtml())
plt.close(animation._fig)
animation_html


## Reading The Animation

- `Original PSD` is the reference campaign frame in linear power.
- `Preprocessing-only` shows the deterministic baseline before the learned codec stage.
- `Codec reconstruction` is the full deployed round-trip through the ONNX encoder and server decoder.
- The highlighted point in the lower-left panel shows where the current frame sits in the evaluated packet-size / distortion trade-off.
- The text panel exposes the exact per-frame packet budget and reconstruction metrics so you can correlate visible spectral errors with deployment costs.
